# Sampling and operator composition

This notebook focuses on sampling operators and on building more complicated linear maps by multiplication of MiniOps operators.

For arrays with time in the first dimension, `trace_sampling_op` samples complete traces. The shorthand `sampling_op(idx, full_size)` performs the same operation when `full_size` has two or more dimensions.

In [1]:
using Pkg
Pkg.activate("..")

using MiniOps
using Random
using LinearAlgebra
using Printf
using CairoMakie
using SeisProcessing

  Activating project at `~/MiniOps`


## 1. Scalar sampling

For a vector, `sampling_op(idx, n)` extracts individual samples. Its adjoint scatters them back into a vector of length `n`.

In [3]:
n = 30
idx = [2, 5, 9]
S = sampling_op(idx, n)

x = collect(1.0:n)
y = S * x
xa = S' * y

@show y xa;

y = [2.0, 5.0, 9.0]
xa = [0.0, 2.0, 0.0, 0.0, 5.0, 0.0, 0.0, 0.0, 9.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


## 2. Whole-trace sampling

For seismic-style arrays, the first dimension is time and all remaining dimensions identify traces. `trace_sampling_op` flattens those trace dimensions using Julia's column-major ordering.

In [4]:
nt, nx, ny, nw, nz = 20, 30, 40, 10, 5
idx = [3, 11, 20]

for full_size in ((nt,nx), (nt,nx,ny), (nt,nx,ny,nw), (nt,nx,ny,nw,nz))
    R = trace_sampling_op(idx, full_size)
    x = randn(full_size...)
    y = R * x
    xa = R' * y
    println("full size = ", full_size,
            "  data size = ", size(y),
            "  adjoint size = ", size(xa))
end

full size = (20, 30)  data size = (20, 3)  adjoint size = (20, 30)
full size = (20, 30, 40)  data size = (20, 3)  adjoint size = (20, 30, 40)
full size = (20, 30, 40, 10)  data size = (20, 3)  adjoint size = (20, 30, 40, 10)
full size = (20, 30, 40, 10, 5)  data size = (20, 3)  adjoint size = (20, 30, 40, 10, 5)


## 3. Composition

If `A`, `B`, and `C` are MiniOps operators, `A * B * C` creates the composite map without forming a matrix. The adjoint reverses the order automatically:


$$(ABC)^* = C^*B^*A^*$$


Here we combine diagonal weighting, trace sampling, and scaling.

In [6]:
nt, nx = 20, 30
idx = [1, 2, 3, 5, 10, 15, 20, 25, 30]

R  = trace_sampling_op(idx, (nt,nx))
D  = diag_op(ones(nt,nx))
Sc = scaling_op(3.0)
A  = Sc * R * D

x = randn(nt,nx)
y = randn(size(A*x))

println("<Ax,y> = ", dot(A*x, y))
println("<x,A'y> = ", dot(x, A'*y))
println("adjoint test = ", adjoint_test(A, x, y))
println("name = ", A.name, ", m = ", A.m, ", n = ", A.n)

<Ax,y> = 38.36462127278897
<x,A'y> = 38.36462127278899
adjoint test = (true, 3.7041561323274125e-16)
name = scaling*trace_sampling*diag, m = -1, n = -1


## 4. Attaching shape information

Some operators are deliberately size-agnostic. `with_shape` returns an equivalent operator carrying shape information from a representative input.

In [7]:
F_unplanned = fft_op()
x = randn(ComplexF64,20,20)
F_shaped = with_shape(F_unplanned, x)

println("before with_shape: m = ", F_unplanned.m, ", n = ", F_unplanned.n)
println("after  with_shape: m = ", F_shaped.m,    ", n = ", F_shaped.n)
println("adjoint test: ", adjoint_test(F_shaped, x, randn(ComplexF64,20,20)))

before with_shape: m = -1, n = -1
after  with_shape: m = 400, n = 400
adjoint test: (true, 4.785809315922984e-16)
